# Evaluation: rule-based fall detector on URFD

In `rule_based.ipynb` I tuned the State Machines detector's thresholds on Le2i.
Le2i was my validation set — the data I looked at while picking my thresholds.
(`V`, `N`, `P`, `angle_thresh`, `bbox_thresh`)

URFD is my held out test set. The rules below were never tuned
against it, so the metrics in this notebook are an honest measure of
how well the detector generalizes.

Two things actually differ between Le2i and URFD:

1. **Framerate.** Le2i = 25 fps. URFD = 30 fps. This matters for
   `hip_velocity` (per-second), and it's the only place I'll change
   anything in the pipeline.
2. **Camera & scene.** Different rooms, different actors, different
   lighting. The thresholds were not picked with any of this in mind.

Plan: keep every rule and threshold byte-identical to `rule_based.ipynb`,
swap in URFD's features, and read off the metrics. If the detector fails
in new ways here, those failures are the motivation for either going back 
and tuning my rule based approach/thresholds or moving to an LSTM in Phase 6.


In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)


In [14]:
# Load the per-frame URFD features extracted by phases/extract_features_urfd.py.
feature_df = pd.read_csv('/Users/aryanbaldua/Anoki_Labs/data/urfd_features.csv')

print('shape:', feature_df.shape)
print('columns:', feature_df.columns.tolist())
feature_df.head()


shape: (11936, 7)
columns: ['scene', 'video', 'true_label', 'frame', 'angle', 'hip_height', 'bbox_ratio']


,scene,video,true_label,frame,angle,hip_height,bbox_ratio
0,URFD_adl,adl-01,ADL,1,NaN,NaN,NaN
1,URFD_adl,adl-01,ADL,2,NaN,NaN,NaN
2,URFD_adl,adl-01,ADL,3,NaN,NaN,NaN
3,URFD_adl,adl-01,ADL,4,NaN,NaN,NaN
4,URFD_adl,adl-01,ADL,5,NaN,NaN,NaN


In [15]:
# Same identifier convention as rule_based.ipynb so the rest of the
# code can be reused unchanged.

# clip_id    — unique label for each frame
# true_binary — 1 for fall, 0 for ADL for each overall video
feature_df['clip_id']     = feature_df['scene'] + '_' + feature_df['video']
feature_df['true_binary'] = (feature_df['true_label'] == 'FALL').astype(int)

print('unique clips:', feature_df['clip_id'].nunique())

print('\nlabel balance across all rows:')
print(feature_df['true_binary'].value_counts())

# Expect: 30 fall clips, 40 adl clips.
print('\nlabel balance across clips:')
print(feature_df.groupby('clip_id')['true_binary'].max().value_counts())


unique clips: 70

label balance across all rows:
true_binary
0    8941
1    2995
Name: count, dtype: int64

label balance across clips:
true_binary
0    40
1    30
Name: count, dtype: int64


In [16]:
feature_df['pose_detected'] = (
    feature_df['angle'].notna() & feature_df['hip_height'].notna()
)

# Per-clip detection rate — fraction of frames where we got a pose.
detection_rate = (
    feature_df.groupby('clip_id')['pose_detected']
              .mean()                          
              .reset_index()
              .rename(columns={'pose_detected': 'detection_rate'})
)

print('Per-clip detection rate stats:')
print(detection_rate['detection_rate'].describe())

print('\nClips with <50% detection (where we lose more frames than we keep):')
print(detection_rate[detection_rate['detection_rate'] < 0.5]
      .sort_values('detection_rate'))


Per-clip detection rate stats:
count    70.000000
mean      0.766113
std       0.178835
min       0.356522
25%       0.666902
50%       0.773950
75%       0.902778
max       1.000000
Name: detection_rate, dtype: float64

Clips with <50% detection (where we lose more frames than we keep):
            clip_id  detection_rate
37  URFD_adl_adl-38        0.356522
21  URFD_adl_adl-22        0.370833
29  URFD_adl_adl-30        0.415000
18  URFD_adl_adl-19        0.428000
22  URFD_adl_adl-23        0.440909
20  URFD_adl_adl-21        0.478571
38  URFD_adl_adl-39        0.492593


In [17]:
# This is the one cell where URFD's pipeline differs from Le2i's 
# as we need to account for the different FPS as it affects our hip velocity threshold.
#

FPS = 30   # URFD videos are 30 fps. Le2i was 25.

# diff() needs the rows in time order within each clip — otherwise we'd
# subtract frame 50 from frame 49 in some random order. sort first.
feature_df = (
    feature_df.sort_values(['clip_id', 'frame'])
              .reset_index(drop=True)
)

feature_df['hip_velocity'] = (
    feature_df.groupby('clip_id')['hip_height'].diff() * FPS
)


print('hip_velocity column added.')
print('\nlargest hip_velocity values seen:')
print(
    feature_df.dropna(subset=['hip_velocity'])
              .nlargest(5, 'hip_velocity')[
                  ['clip_id', 'frame', 'hip_height', 'hip_velocity']
              ]
)
print('\nIt is strange that the highest hip velocity were from ADL clips')
print('Also a hip velocity of near 6 seems unreasonable so will need to investigate')


hip_velocity column added.

largest hip_velocity values seen:
                 clip_id  frame  hip_height  hip_velocity
476      URFD_adl_adl-03    147      0.7126         5.991
474      URFD_adl_adl-03    145      0.7092         5.919
574      URFD_adl_adl-04     65      0.4516         4.332
10588  URFD_fall_fall-13     14      0.7060         3.990
8200     URFD_adl_adl-38    205      0.7497         3.978

It is strange that the highest hip velocity were from ADL clips
Also a hip velocity of near 6 seems unreasonable so will need to investigate


In [18]:
# State machine

# The three states:
#   state 0 (idle) - waiting for any downward velocity burst
#   state 1 (saw descent) - burst happened
#   state 2 (confirming) - posture seen at least once

def detect_fall_in_clip_or(group, V, N, P, max_gap, angle_thresh, bbox_thresh):
    # Sort by frame so the SM steps through time in order, not in DataFrame storage order
    g = group.sort_values('frame').reset_index(drop=True)

    state           = 0   # current SM State
    wait_counter    = 0   # frames spent in state 1
    persist_counter = 0   # frames posture has held in state 2
    gap_counter    = 0    # consecutive non-posture frames inside state 2

    for _, row in g.iterrows():
        vel   = row['hip_velocity']
        angle = row['angle']
        bbox  = row['bbox_ratio']

        posture = (
            (pd.notna(angle) and angle >= angle_thresh) or
            (pd.notna(bbox)  and bbox  >  bbox_thresh)
        )
        descent = pd.notna(vel) and vel > V

        if state == 0:
            if descent:
                state, wait_counter = 1, 0

        elif state == 1:
            wait_counter += 1
            if posture:
                state, persist_counter, gap_counter = 2, 1, 0
            elif wait_counter >= N:
                state = 0 

        elif state == 2:
            if posture:
                persist_counter += 1
                gap_counter = 0
                if persist_counter >= P:
                    return 1   # FALL
            else:
                gap_counter += 1
                if gap_counter > max_gap:
                    state, persist_counter, gap_counter = 0, 0, 0

    return 0


In [19]:
import itertools, time
import numpy as np

# Acknowledging that rules/thresholds should not change for 
# evaluation set but wanted to see if I could weed out false 
# positives. Different dataset seems to need different thresholds

# Precompute each clip's signals as numpy arrays once.
clip_data = []
for clip_id, group in feature_df.groupby('clip_id'):
    g = group.sort_values('frame')
    clip_data.append((
        clip_id,
        int(g['true_binary'].max()),
        g['hip_velocity'].to_numpy(dtype=np.float64),
        g['angle'].to_numpy(dtype=np.float64),
        g['bbox_ratio'].to_numpy(dtype=np.float64),
    ))

# State machine that takes precomputed boolean arrays. Same logic as
# detect_fall_in_clip_or, but no NaN checks or float compares in the hot loop.
def run_sm_bool(descent, posture, N, P, max_gap):
    state = wait_counter = persist_counter = gap_counter = 0
    n = len(descent)
    for i in range(n):
        if state == 0:
            if descent[i]:
                state, wait_counter = 1, 0
        elif state == 1:
            wait_counter += 1
            if posture[i]:
                state, persist_counter, gap_counter = 2, 1, 0
            elif wait_counter >= N:
                state = 0
        else:
            if posture[i]:
                persist_counter += 1
                gap_counter = 0
                if persist_counter >= P:
                    return 1
            else:
                gap_counter += 1
                if gap_counter > max_gap:
                    state = persist_counter = gap_counter = 0
    return 0

print(f'Prepared {len(clip_data)} clips.')

Prepared 70 clips.


In [20]:
# Floats: finer fractional steps.
# Integers (N, P, max_gap): step 1 is the finest meaningful step.
V_grid            = np.round(np.arange(0.15, 0.51, 0.025), 4).tolist()  # 0.025 steps
N_grid            = list(range(15, 46, 5))                              # int, step 5
P_grid            = list(range(2, 21, 1))                               # int, step 1
max_gap_grid      = list(range(6, 19, 2))                               # int, step 2
angle_thresh_grid = np.round(np.arange(45, 75.1, 2.5), 2).tolist()      # 2.5 steps
bbox_thresh_grid  = np.round(np.arange(0.85, 1.51, 0.05), 4).tolist()   # 0.05 steps

total = (len(V_grid) * len(N_grid) * len(P_grid) *
         len(max_gap_grid) * len(angle_thresh_grid) * len(bbox_thresh_grid))
print(f'V          ({len(V_grid):>3}): {V_grid}')
print(f'N          ({len(N_grid):>3}): {N_grid}')
print(f'P          ({len(P_grid):>3}): {P_grid}')
print(f'max_gap    ({len(max_gap_grid):>3}): {max_gap_grid}')
print(f'angle      ({len(angle_thresh_grid):>3}): {angle_thresh_grid}')
print(f'bbox       ({len(bbox_thresh_grid):>3}): {bbox_thresh_grid}')
print(f'\nTotal configurations: {total:,}')

V          ( 15): [0.15, 0.175, 0.2, 0.225, 0.25, 0.275, 0.3, 0.325, 0.35, 0.375, 0.4, 0.425, 0.45, 0.475, 0.5]
N          (  7): [15, 20, 25, 30, 35, 40, 45]
P          ( 19): [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
max_gap    (  7): [6, 8, 10, 12, 14, 16, 18]
angle      ( 13): [45.0, 47.5, 50.0, 52.5, 55.0, 57.5, 60.0, 62.5, 65.0, 67.5, 70.0, 72.5, 75.0]
bbox       ( 14): [0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.25, 1.3, 1.35, 1.4, 1.45, 1.5]

Total configurations: 2,541,630


In [21]:
rows = []
tic = time.time()

# Outer loop: (V, angle, bbox). Compute descent and posture once per combo.
# Inner loop: (N, P, max_gap). Just runs the state machine on precomputed bools.
outer_combos = list(itertools.product(V_grid, angle_thresh_grid, bbox_thresh_grid))
inner_combos = list(itertools.product(N_grid, P_grid, max_gap_grid))
print(f'Outer: {len(outer_combos):,}   Inner: {len(inner_combos):,}')

report_every = max(1, len(outer_combos) // 20)

for outer_idx, (V, at, bt) in enumerate(outer_combos):
    # Precompute booleans for all clips at this (V, at, bt).
    # NaN comparisons return False in numpy, so these are NaN-safe.
    with np.errstate(invalid='ignore'):
        precomputed = []
        for _cid, true_label, vel, angle, bbox in clip_data:
            descent = (vel > V).tolist()
            posture = ((angle >= at) | (bbox > bt)).tolist()
            precomputed.append((true_label, descent, posture))

    for N, P, mg in inner_combos:
        tp = fp = tn = fn = 0
        for true_label, d, p in precomputed:
            pred = run_sm_bool(d, p, N, P, mg)
            if   true_label == 1 and pred == 1: tp += 1
            elif true_label == 0 and pred == 1: fp += 1
            elif true_label == 0 and pred == 0: tn += 1
            else:                               fn += 1

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall    = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        rows.append({
            'V': V, 'N': N, 'P': P, 'max_gap': mg, 'angle': at, 'bbox': bt,
            'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
            'precision': precision, 'recall': recall, 'f1': f1,
            'accuracy': (tp + tn) / (tp + tn + fp + fn),
        })

    if (outer_idx + 1) % report_every == 0:
        elapsed = time.time() - tic
        eta = elapsed / (outer_idx + 1) * (len(outer_combos) - outer_idx - 1)
        print(f'  outer {outer_idx+1:>5,}/{len(outer_combos):,}  '
              f'elapsed={elapsed:>6.1f}s  ETA={eta:>6.1f}s')

grid_results = pd.DataFrame(rows)
print(f'\nDone in {time.time() - tic:.1f}s. Results: {len(grid_results):,} rows.')

Outer: 2,730   Inner: 931
  outer   136/2,730  elapsed=  30.0s  ETA= 571.7s
  outer   272/2,730  elapsed=  61.4s  ETA= 555.3s
  outer   408/2,730  elapsed=  91.7s  ETA= 521.6s
  outer   544/2,730  elapsed= 121.7s  ETA= 489.1s
  outer   680/2,730  elapsed= 150.3s  ETA= 453.0s
  outer   816/2,730  elapsed= 178.6s  ETA= 418.9s
  outer   952/2,730  elapsed= 206.7s  ETA= 386.0s
  outer 1,088/2,730  elapsed= 235.4s  ETA= 355.3s
  outer 1,224/2,730  elapsed= 262.8s  ETA= 323.4s
  outer 1,360/2,730  elapsed= 290.2s  ETA= 292.4s
  outer 1,496/2,730  elapsed= 317.4s  ETA= 261.8s
  outer 1,632/2,730  elapsed= 344.4s  ETA= 231.7s
  outer 1,768/2,730  elapsed= 371.1s  ETA= 201.9s
  outer 1,904/2,730  elapsed= 397.8s  ETA= 172.6s
  outer 2,040/2,730  elapsed= 424.5s  ETA= 143.6s
  outer 2,176/2,730  elapsed= 450.9s  ETA= 114.8s
  outer 2,312/2,730  elapsed= 476.8s  ETA=  86.2s
  outer 2,448/2,730  elapsed= 790.1s  ETA=  91.0s
  outer 2,584/2,730  elapsed= 937.2s  ETA=  53.0s
  outer 2,720/2,730  ela

In [32]:
top_f1 = (
    grid_results.sort_values(['f1', 'recall', 'precision'],
                             ascending=[False, False, False])
                .head(5).reset_index(drop=True)
)
print('Top 5 by F1:')
print(top_f1.to_string(index=False))

RECALL_FLOOR = 0.8
high_recall = grid_results[grid_results['recall'] >= RECALL_FLOOR]
top_precision = (
    high_recall.sort_values(['precision', 'recall'], ascending=[False, False])
               .head(5).reset_index(drop=True)
)
print(f'\nTop 5 by precision (recall >= {RECALL_FLOOR}):')
print(top_precision.to_string(index=False))

Top 5 by F1:
  V  N  P  max_gap  angle  bbox  tn  fp  fn  tp  precision  recall       f1  accuracy
0.4 15  3        6   75.0  1.15  31   9   0  30   0.769231     1.0 0.869565  0.871429
0.4 15  3        6   75.0  1.20  31   9   0  30   0.769231     1.0 0.869565  0.871429
0.4 15  2        6   75.0  1.25  31   9   0  30   0.769231     1.0 0.869565  0.871429
0.4 15  3        6   75.0  1.25  31   9   0  30   0.769231     1.0 0.869565  0.871429
0.4 15  3       18   75.0  1.25  31   9   0  30   0.769231     1.0 0.869565  0.871429

Top 5 by precision (recall >= 0.8):
  V  N  P  max_gap  angle  bbox  tn  fp  fn  tp  precision  recall       f1  accuracy
0.4 15  3        6   75.0  1.15  31   9   0  30   0.769231     1.0 0.869565  0.871429
0.4 15  3        6   75.0  1.20  31   9   0  30   0.769231     1.0 0.869565  0.871429
0.4 15  2        6   75.0  1.25  31   9   0  30   0.769231     1.0 0.869565  0.871429
0.4 15  3        6   75.0  1.25  31   9   0  30   0.769231     1.0 0.869565  0.871429
0.4 

In [35]:
"""
# original values carried over from Le2i
FINAL_V       = 0.215   
FINAL_N       = 30      
FINAL_P       = 2       
FINAL_MAX_GAP = 12      
FINAL_ANGLE   = 45      
FINAL_BBOX    = 0.85    
"""

# new values from grid search
FINAL_V       = 0.4   
FINAL_N       = 15     
FINAL_P       = 3       
FINAL_MAX_GAP = 6      
FINAL_ANGLE   = 75      
FINAL_BBOX    = 1.15  

# Ground truth at the clip level — one label per video.
clip_truth = (
    feature_df.groupby('clip_id')['true_binary']
              .max()               
              .reset_index()
              .rename(columns={'true_binary': 'true_clip'})
)

# Run the SM once per clip and collect (clip_id, prediction).
preds = []
for clip_id, group in feature_df.groupby('clip_id'):
    pred = detect_fall_in_clip_or(
        group,
        V           = FINAL_V,
        N           = FINAL_N,
        P           = FINAL_P,
        max_gap     = FINAL_MAX_GAP,
        angle_thresh= FINAL_ANGLE,
        bbox_thresh = FINAL_BBOX,
    )
    preds.append({'clip_id': clip_id, 'pred_clip': pred})

# Join predictions to ground truth. One row per clip, two columns we care
# about: pred_clip and true_clip
results = pd.DataFrame(preds).merge(clip_truth, on='clip_id')

print(f'Ran detector on {len(results)} clips.')
print(f'Predicted FALL: {(results["pred_clip"] == 1).sum()}')
print(f'Predicted ADL : {(results["pred_clip"] == 0).sum()}')
results.head()


Ran detector on 70 clips.
Predicted FALL: 39
Predicted ADL : 31


,clip_id,pred_clip,true_clip
0,URFD_adl_adl-01,0,0
1,URFD_adl_adl-02,0,0
2,URFD_adl_adl-03,0,0
3,URFD_adl_adl-04,0,0
4,URFD_adl_adl-05,0,0


In [36]:
y_true = results['true_clip']
y_pred = results['pred_clip']

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()   

print('URFD held-out test — rule-based detector with Le2i-tuned thresholds')
print('              pred_nofall  pred_fall')
print(f'true_nofall   {tn:>11}  {fp:>9}')
print(f'true_fall     {fn:>11}  {tp:>9}')
print()
print(f'accuracy : {accuracy_score(y_true, y_pred):.3f}')
print(f'precision: {precision_score(y_true, y_pred, zero_division=0):.3f}')
print(f'recall   : {recall_score(y_true, y_pred, zero_division=0):.3f}')
print(f'f1       : {f1_score(y_true, y_pred, zero_division=0):.3f}')


URFD held-out test — rule-based detector with Le2i-tuned thresholds
              pred_nofall  pred_fall
true_nofall            31          9
true_fall               0         30

accuracy : 0.871
precision: 0.769
recall   : 1.000
f1       : 0.870


In [37]:
# Pull the 23 ADL clips that the detector wrongly flagged as falls.
# These are the clips where true_clip == 0 (ADL) but pred_clip == 1 (fall).
fps = results[(results['true_clip'] == 0) & (results['pred_clip'] == 1)]
print(f'False positives: {len(fps)}')

# For each FP clip, pull the actual feature signals so we can see WHY it fired.
# The five columns we care about:

#   detection_rate  
#   max_angle     
#   max_bbox    
#   max_velocity    — anything > ~4 is almost certainly a MediaPipe tracking glitch, not real motion.
#   frames_either   — numbers of frames posture rules fire (angle or bbox)

fp_details = []
for cid in fps['clip_id']:
    clip = feature_df[feature_df['clip_id'] == cid]
    fp_details.append({
        'clip_id'        : cid,
        'detection_rate' : clip['pose_detected'].mean(),
        'max_angle'      : clip['angle'].max(),
        'max_bbox'       : clip['bbox_ratio'].max(),
        'max_velocity'   : clip['hip_velocity'].max(),
        'frames_angle_ok': (clip['angle']      >= FINAL_ANGLE).sum(),
        'frames_bbox_ok' : (clip['bbox_ratio'] >  FINAL_BBOX ).sum(),
        'frames_either'  : ((clip['angle']      >= FINAL_ANGLE) |
                            (clip['bbox_ratio'] >  FINAL_BBOX )).sum(),
    })

# Sort by max_velocity — clips at the top are likely glitch-driven FPs
fp_details_df = (
    pd.DataFrame(fp_details)
      .sort_values('max_velocity', ascending=False)
      .reset_index(drop=True)
)

print()
print(fp_details_df.to_string(index=False))


False positives: 9

        clip_id  detection_rate  max_angle  max_bbox  max_velocity  frames_angle_ok  frames_bbox_ok  frames_either
URFD_adl_adl-38        0.356522   179.9381    1.3468         3.978               53               1             53
URFD_adl_adl-30        0.415000   174.8926    1.4359         2.319               61               4             61
URFD_adl_adl-40        0.527273   175.4569    1.0308         2.163               82               0             82
URFD_adl_adl-39        0.492593   179.9106    0.8975         1.776               39               0             39
URFD_adl_adl-10        0.933333    60.7390    1.7207         1.380                0              74             74
URFD_adl_adl-32        0.810000   173.9587    2.3655         1.125               27              25             44
URFD_adl_adl-34        0.795812   179.9425    1.4649         1.047               99              18            100
URFD_adl_adl-31        0.844000   179.3973    1.5006        

Going through each false positive and explaining why my state machine based approach failed. Trying to see 
if there is a justification for using LSTM


** This is for the original (not optimized) thresholds *
URFD_adl_adl-03 - LSTM should fix. squatting movement
URFD_adl_adl-04 - LSTM should fix. bending down 
URFD_adl_adl-38 - LSTM should fix. dropping to do a pushup
URFD_adl_adl-30 - LSTM should fix. lying down on floor
URFD_adl_adl-40 - LSTM should fix. lying down on floor
URFD_adl_adl-15 - LSTM should fix. bending over
URFD_adl_adl-33 - LSTM should fix. lying down
URFD_adl_adl-39 - LSTM should fix. lying down
URFD_adl_adl-23 - LSTM should fix. sitting + lying down quickly
URFD_adl_adl-17 - LSTM should fix. bending over
URFD_adl_adl-10 - LSTM should fix. sitting + lying down quickly
URFD_adl_adl-32 - LSTM should fix. dropping to floor to pick up item
URFD_adl_adl-34 - LSTM should fix. dropping to floor to pick up item
URFD_adl_adl-31 - LSTM should fix. dropping to floor to pick up item
URFD_adl_adl-11 - LSTM should fix. sitting + lying down quickly
URFD_adl_adl-19 - LSTM should fix. sitting down quickly
URFD_adl_adl-22 - LSTM should fix. sitting + lying down quickly
URFD_adl_adl-35 - LSTM should fix. dropping to floor to pick up item
URFD_adl_adl-37 - LSTM should fix. sitting + lying down quickly
URFD_adl_adl-36 - LSTM should fix. sitting + lying down quickly
URFD_adl_adl-05 - LSTM should fix. bending over
URFD_adl_adl-09 - LSTM should fix. sitting down
URFD_adl_adl-06 - LSTM should fix. bending over


Based on the false postives and the reason for them occuring, I believe an LSTM 
appoach will help solve a lot of incorrect predictions. Actions like sitting, 
sleeping, and bending over should be resolved, although actions like dropping to 
the floor to pick up an item may be quite hard for an LSTM to even catch.
    